In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import regex as re
import sys
import spotipy
import spotipy.util as util

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
from spotipy.oauth2 import SpotifyClientCredentials
from spotipy.oauth2 import SpotifyOAuth

In [ ]:
data = pd.read_csv("spotifydataset.csv")
db = data.copy().drop(columns=["Unnamed: 0","mode","key","duration_ms","time_signature"])

consolodatedgenre = db.groupby("track_id")["track_genre"].apply(list).reset_index()
db = db.merge(consolodatedgenre[["track_id","track_genre"]], on="track_id",how="left").drop(columns="track_genre_x").rename(columns={"track_genre_y":"track_genre"})
db = db.drop_duplicates(subset=['track_id'])


db['popularity_2'] = db['popularity'].apply(lambda x: int(x/5))
db = db.drop(columns="popularity")
floats = db[["energy","loudness","speechiness","acousticness","instrumentalness","liveness","valence","tempo","danceability"]].reset_index(drop = True)
scaler = MinMaxScaler()
floats_scaled = pd.DataFrame(scaler.fit_transform(floats), columns = floats.columns) * .5
db[["energy","loudness","speechiness","acousticness","instrumentalness","liveness","valence","tempo","danceability"]] = floats_scaled

In [ ]:
client_id = 'CLIENT_ID'
client_secret= 'CLIENT_SECRET'
redirecturi = "redirecturi"

In [ ]:
auth = SpotifyOAuth(
    client_id=client_id,
    client_secret=client_secret,
    redirect_uri=redirecturi,
    scope="user-library-read",
    open_browser=False
)

url = auth.get_authorize_url()
print(url)

In [ ]:
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(client_id=client_id,client_secret=client_secret,redirect_uri=redirecturi,scope="user-library-read",open_browser=False))   
current_user = sp.current_user()

In [ ]:
id_name = {}
list_photo = {}
for i in sp.current_user_playlists()['items']:

    id_name[i['name']] = i['uri'].split(':')[2]
    list_photo[i['uri'].split(':')[2]] = i['images'][0]['url']

In [ ]:
def ohe(data,col):
    
    ohe = OneHotEncoder()
    cat_data = ohe.fit(data[[col]])
    
    cat_data = ohe.transform(data[[col]]).toarray()

    cat_df = pd.DataFrame(data=cat_data, columns = ohe.get_feature_names_out(),index=data.index)
   
    return data.join(cat_df).drop(columns=col)


def tfidf(data,col):
    tfidf = TfidfVectorizer()
    tfidf_matrix =  tfidf.fit_transform(data[col].apply(lambda x: " ".join(x)))
    genre_df = pd.DataFrame(tfidf_matrix.toarray())
    genre_df.columns = ['genre' + ":" + i for i in tfidf.get_feature_names_out()]
    genre_df.reset_index(drop = True, inplace=True)
    return data.join(genre_df).drop(columns="track_genre")

In [ ]:
db_ohe = ohe(db,"popularity_2")

tracknames = tfidf(db_ohe,"track_genre")

db_ohe_tn = tracknames.drop(columns={"track_name","artists","album_name"})

In [ ]:
def recmachine(playlistname,data):
    
    
    idarray = playlisttrackidlist(id_name[playlistname])
    iddf = pd.DataFrame({"col": idarray})
    sortedarray = iddf[iddf["col"].isin(data["track_id"])]
    
    if sortedarray.shape[0] < 1:
        return print("no songs matching in the dataset!")
    else:
 
        Y = data[data["track_id"].isin(sortedarray["col"])].drop(columns="track_id").fillna(0)
        Yindex=Y.index.tolist()
        X=data.drop(Yindex).drop(columns="track_id").fillna(0)
        sim = cosine_similarity(X,Y)
    
    
        X["simscore"] = sim.sum(axis=1)  
        indextotrackname = tracknames.drop(Yindex)[["track_name","artists","album_name"]]
        newdata = X.join(indextotrackname).sort_values("simscore",ascending=False).iloc[:,-4:]
        return newdata.head(20)
        

In [ ]:
track_ids=pd.DataFrame(db_ohe_tn["track_id"])

PCAdata = db_ohe_tn.drop(columns="track_id").fillna(0).astype('float')

u, s, vt = np.linalg.svd(PCAdata,full_matrices=False)


db_PCA = np.dot(PCAdata,np.transpose(vt[:15]))
db_PCA_clean = pd.DataFrame(db_PCA).fillna(0)
db_PCA_id = track_ids.join(db_PCA_clean)